In [ ]:
%run "/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/code/Initialization/InitializationTime.ipynb"

In [ ]:
df.columns

In [ ]:
df = df[["Sentiment Natural Log","Negative Natural Log", "VIX Natural Log","Positive Natural Log","Polarity","WordCount Natural Log","Type"]]

In [ ]:
df.reset_index(inplace=True)

In [ ]:
df = df.groupby("Type").sum()

In [ ]:
df["Sentiment Natural Log"]

In [ ]:
# df["Sentiment Natural Log"].plot(kind="hist")

In [ ]:
df = df.transpose()

In [ ]:
df["Type"] = df.index

In [ ]:
df

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
# from bioinfokit.analys import get_data
import numpy as np
import pandas as pd
# load dataset as pandas dataframe
# df = get_data('gexp').data
df.head(2)

In [ ]:
df = df.iloc[: , :6]

In [ ]:
# this is an optional step
df_st =  StandardScaler().fit_transform(df)  
pd.DataFrame(df_st, columns=df.columns)#.head(2)
# output

# Perform PCA using scikit-learn

In [ ]:
pca_out = PCA().fit(df_st)

# get the component variance
# Proportion of Variance (from PC1 to PC6)
pca_out.explained_variance_ratio_
# output
# array([0.2978742 , 0.27481252, 0.23181442, 0.19291638, 0.00144353,
#        0.00113895])

# Cumulative proportion of variance (from PC1 to PC6)   
np.cumsum(pca_out.explained_variance_ratio_)
# output
# array([0.2978742 , 0.57268672, 0.80450114, 0.99741752, 0.99886105,
#        1.        ])
       
# component loadings or weights (correlation coefficient between original variables and the component) 
# component loadings represents the elements of the eigenvector
# the squared loadings within the PCs always sums to 1
loadings = pca_out.components_
num_pc = pca_out.n_features_
pc_list = ["PC"+str(i) for i in list(range(1, num_pc+1))]
loadings_df = pd.DataFrame.from_dict(dict(zip(pc_list, loadings)))
loadings_df['variable'] = df.columns.values
loadings_df = loadings_df.set_index('variable')
loadings_df
# output
#                PC1       PC2       PC3       PC4       PC5       PC6
# variable
# A        -0.510898  0.452234  0.227356 -0.323464  0.614881  0.008372
# B        -0.085908  0.401197  0.708556  0.132788 -0.558448 -0.010616
# C         0.477477 -0.100994  0.462437  0.487951  0.556605  0.007893
# D         0.370318  0.611485 -0.308295  0.054973 -0.007642  0.625159
# E         0.568491  0.300118 -0.011775 -0.484115  0.009382 -0.593425
# F         0.208090 -0.400426  0.370440 -0.634234 -0.010111  0.506732

# positive and negative values in component loadings reflects the positive and negative 
# correlation of the variables with the PCs. Except A and B, all other variables have 
# positive projection on first PC.

# get correlation matrix plot for loadings
import seaborn as sns
import matplotlib.pyplot as plt
ax = sns.heatmap(loadings_df, annot=True, cmap='Spectral')
plt.show()

# Principal component (PC) retention

In [ ]:
import scipy
print(scipy.__version__)

In [ ]:
import bioinfokit
bioinfokit.__version__

In [ ]:
# get eigenvalues (variance explained by each PC)  
pca_out.explained_variance_
# output
# array([1.78994905, 1.65136965, 1.39299071, 1.15924943, 0.0086743 ,
#        0.00684401])
# get scree plot (for scree or elbow test)
from bioinfokit.visuz import cluster
cluster.screeplot(obj=[pc_list, pca_out.explained_variance_ratio_])
# Scree plot will be saved in the same directory with name screeplot.png

# PCA loadings plots

In [ ]:
# get PCA loadings plots (2D and 3D)
# 2D
cluster.pcaplot(x=loadings[0], y=loadings[1], labels=df.columns.values, 
    var1=round(pca_out.explained_variance_ratio_[0]*100, 2),
    var2=round(pca_out.explained_variance_ratio_[1]*100, 2))

# 3D
cluster.pcaplot(x=loadings[0], y=loadings[1], z=loadings[2],  labels=df.columns.values, 
    var1=round(pca_out.explained_variance_ratio_[0]*100, 2), var2=round(pca_out.explained_variance_ratio_[1]*100, 2), 
    var3=round(pca_out.explained_variance_ratio_[2]*100, 2))

# PCA biplot

In [ ]:
# get PC scores
pca_scores = PCA().fit_transform(df)

# get 2D biplot
cluster.biplot(cscore=pca_scores, loadings=loadings, labels=df.columns.values, var1=round(pca_out.explained_variance_ratio_[0]*100, 2),
    var2=round(pca_out.explained_variance_ratio_[1]*100, 2))
    
# get 3D biplot
cluster.biplot(cscore=pca_scores, loadings=loadings, labels=df.columns.values, 
    var1=round(pca_out.explained_variance_ratio_[0]*100, 2), var2=round(pca_out.explained_variance_ratio_[1]*100, 2), 
    var3=round(pca_out.explained_variance_ratio_[2]*100, 2))

# PCA interpretation

The first three PCs (3D) contribute ~81% of the total variation in the dataset and have eigenvalues > 1, and thus provides a good approximation of the variation present in the original 6D dataset (see the cumulative proportion of variance and scree plot). The cut-off of cumulative 70% variation is common to retain the PCs for analysis (Jolliffe et al., 2016). Even though the first four PCs contribute ~99% and have eigenvalues > 1, it will be difficult to visualize them at once and needs to perform pairwise visualization.
From the biplot and loadings plot, we can see the variables D and E are highly associated and forms cluster (gene expression response in D and E conditions are highly similar). Similarly, A and B are highly associated and forms another cluster (gene expression response in A and B conditions are highly similar but different from other clusters). If the variables are highly associated, the angle between the variable vectors should be as small as possible in the biplot.
The length of PCs in biplot refers to the amount of variance contributed by the PCs. The longer the length of PC, the higher the variance contributed and well represented in space.

# Principal component analysis (PCA) with a target variable

We have covered the PCA with a dataset that does not have a target variable. Now, we will perform the PCA on the iris plant dataset, which has a target variable.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
# from bioinfokit.analys import get_data
from bioinfokit.visuz import cluster

# load iris dataset
# df = get_data('iris').data
df.head(2)
# output 
# class (type of iris plant) is target variable
#    sepal_length  sepal_width  petal_length  petal_width        class
# 0           5.1          3.5           1.4          0.2  Iris-setosa
# 1           4.9          3.0           1.4          0.2  Iris-setosa

X = df.iloc[:,0:4]
target = df['ACQUI'].to_numpy()
X.head(2)
#    sepal_length  sepal_width  petal_length  petal_width
# 0           5.1          3.5           1.4          0.2
# 1           4.9          3.0           1.4          0.2

X_st =  StandardScaler().fit_transform(X)
pca_out = PCA().fit(X_st)

# component loadings
loadings = pca_out.components_
loadings
# array([[ 0.52237162, -0.26335492,  0.58125401,  0.56561105],
#        [ 0.37231836,  0.92555649,  0.02109478,  0.06541577],
#        [-0.72101681,  0.24203288,  0.14089226,  0.6338014 ],
#        [-0.26199559,  0.12413481,  0.80115427, -0.52354627]])

# get eigenvalues (variance explained by each PC)  
pca_out.explained_variance_
# array([2.93035378, 0.92740362, 0.14834223, 0.02074601])

# get biplot
pca_scores = PCA().fit_transform(X_st)
cluster.biplot(cscore=pca_scores, loadings=loadings, labels=X.columns.values, var1=round(pca_out.explained_variance_ratio_[0]*100, 2),
    var2=round(pca_out.explained_variance_ratio_[1]*100, 2), colorlist=target)

# PCA from scratch

Computing the PCA from scratch involves various steps, including standardization of the input dataset (optional step), calculating mean adjusted matrix, covariance matrix, and calculating eigenvectors and eigenvalues.

## Calculate mean adjusted matrix

In [ ]:
# from bioinfokit.analys import get_data
from sklearn.preprocessing import StandardScaler
import pandas as pd
# load iris dataset
# df = get_data('iris').data
df.head(2)
#    sepal_length  sepal_width  petal_length  petal_width        class
# 0           5.1          3.5           1.4          0.2  Iris-setosa
# 1           4.9          3.0           1.4          0.2  Iris-setosa
# the iris dataset has 150 samples (n) and 4 variables (p), i.e., nxp matrix

X = df.iloc[:,0:4]  # p variables
target = df['ACQUI'].to_numpy()  # target variable

# standardize the dataset (this is an optional step)
# I am using this step to get consistent output as per the PCA method used above
X_st =  StandardScaler().fit_transform(X)
X_st = pd.DataFrame(X_st)
X_st.columns = X.columns

# create mean adjusted matrix (subtract each column mean by its value)
X_adjusted = X_st - X_st.mean()
X_adjusted.head(2)
#    sepal_length  sepal_width  petal_length  petal_width
# 0     -0.900681     1.032057     -1.341272    -1.312977
# 1     -1.143017    -0.124958     -1.341272    -1.312977

## Calculate the covariance matrix

In [ ]:
X_adjusted.cov()
#               sepal_length  sepal_width  petal_length  petal_width
# sepal_length      1.006711    -0.110103      0.877605     0.823443
# sepal_width      -0.110103     1.006711     -0.423338    -0.358937
# petal_length      0.877605    -0.423338      1.006711     0.969219
# petal_width       0.823443    -0.358937      0.969219     1.006711

## Eigendecomposition of the covariance matrix

Eigendecomposition of covariance matrix yields eigenvectors (PCs) and eigenvalues (variance of PCs). The elements of eigenvectors are known as loadings. This step involves linear algebra and can be performed using NumPy linalg.eig function. This is a very important step in PCA.

In [ ]:
from numpy.linalg import eig
eigenvalues, eigenvectors = eig(X_adjusted.cov())
eigenvalues
# array([2.93035378, 0.92740362, 0.14834223, 0.02074601])
# we are interested in highest eigenvalues as it explains most of the variance
# this helps to reduce the dimensions

eigenvectors
# array([[ 0.52237162, -0.37231836, -0.72101681,  0.26199559],
#        [-0.26335492, -0.92555649,  0.24203288, -0.12413481],
#        [ 0.58125401, -0.02109478,  0.14089226, -0.80115427],
#        [ 0.56561105, -0.06541577,  0.6338014 ,  0.52354627]])

# column eigenvectors[:,i] is the eigenvectors of eigenvalues eigenvalues[i]